# Parallel Production Training & Probing — `pendulum_mess4`

Parallel multi-horizon training (`k=1`, `k=n//2`, `k=n`) and 6-mode linear probing across 25 pre-validated pendulum configurations.
Spawns standalone worker processes (`_worker.py`) managed via atomic file-locked job queue.

> **Note**: Linear probing uses `torch.linalg.solve` (GPU-accelerated closed-form ridge) with `joblib`-parallelised GroupKFold CV folds and alpha sweep. `StandardScaler` applied before solving.

In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path

# Toggle SMOKE mode (True for smoke testing, False for 600M production training)
SMOKE = False

# --- Production Sweep Hyperparameters ---
SEEDS = [0, 1, 2]
WIDTHS = [32, 64, 128]
N_LAYERS = 4
N_HEADS = 1
BATCH_SIZE = 512
TOTAL_TOKENS = 1000_000_000
N_PROBE_TRAJECTORIES = 1024
CV_FOLDS = 5
RIDGE_ALPHAS = (1.0, 10.0, 50.0)
TEST_FRACTION = 0.2
LOG_EVERY = 50

if SMOKE:
    SEEDS = [0]
    WIDTHS = [32]
    BATCH_SIZE = 16
    TOTAL_TOKENS = 500_000
    N_PROBE_TRAJECTORIES = 64
    RIDGE_ALPHAS = (1.0, 10.0)

# --- Resolve Root & Environment ---
ROOT = Path.cwd()
for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (candidate / 'physics' / 'messk.py').exists():
        ROOT = candidate.resolve()
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch, multiprocessing
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
GPU_COUNT = torch.cuda.device_count() if torch.cuda.is_available() else 0
CPU_CORES = multiprocessing.cpu_count()
NUM_WORKERS = 2
if SMOKE:
    NUM_WORKERS = 2

OUTPUT_DIR = ROOT / 'experiments' / 'pendulum-sweeps' / 'parallel_output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"ROOT={ROOT}\nDEVICE={DEVICE} (GPUs={GPU_COUNT}, CPUs={CPU_CORES})\nWORKERS={NUM_WORKERS}\nSMOKE={SMOKE}\nOUTPUT_DIR={OUTPUT_DIR}")


### Cell 2: Load Physics Configurations
Loads all 25 rows from `pendulum_range.csv` and sets sequence length ($m = 500 // n$). For $n=15$, $m=33$ ($seq\_len=495$).

In [ ]:
import pandas as pd

RANGE_CSV = ROOT / 'experiments' / 'pendulum-sweeps' / 'pendulum_range.csv'
range_df = pd.read_csv(RANGE_CSV)

def config_name(row):
    return f"dt{row['dt']:g}_gamma{row['damping_value']:g}_dv{row['delta_v']:g}_n{int(row['n_steps'])}"

def compute_m(n_steps):
    return 500 // n_steps  # n=15 → m=33, seq_len=495

configs = []
for _, row in range_df.iterrows():
    n = int(row['n_steps'])
    m = compute_m(n)
    configs.append({
        'config_name': config_name(row),
        'delta_v': float(row['delta_v']),
        'gamma': float(row['damping_value']),
        'dt': float(row['dt']),
        'n_steps': n,
        'm': m,
        'seq_len': m * n,
    })

print(f"Loaded {len(configs)} physics configurations from {RANGE_CSV.name}")
display(pd.DataFrame(configs).head(10))


### Cell 3: Build Job Queue
Creates training jobs across physics configurations, model widths, and seeds.

In [ ]:
jobs = []
for cfg in configs:
    for width in WIDTHS:
        for seed in SEEDS:
            job_id = f"{cfg['config_name']}_d{width}_seed{seed}"
            horizons = [1, max(1, cfg['n_steps'] // 2), cfg['n_steps']]
            jobs.append({
                **cfg,
                'width': width,
                'seed': seed,
                'job_id': job_id,
                'horizons': horizons,
            })

print(f"Total jobs: {len(jobs)} (each trains 3 horizons = {len(jobs) * 3} model trainings)")


### Cell 4: Filter Completed Jobs
Checks `pipeline_state.json` to enable incremental execution and resuming.

In [ ]:
state_path = OUTPUT_DIR / 'pipeline_state.json'
completed = set()
if state_path.exists():
    completed = set(json.loads(state_path.read_text()).get('completed', []))

pending = [j for j in jobs if j['job_id'] not in completed]
print(f"Completed: {len(completed)} | Pending: {len(pending)} | Total: {len(jobs)}")


### Cell 5: Spawn Worker Subprocesses
Writes pending jobs to `_job_queue.json` and launches `NUM_WORKERS` parallel `_worker.py` processes.

In [ ]:
if not pending:
    print("All jobs complete. Nothing to do.")
else:
    queue_path = OUTPUT_DIR / '_job_queue.json'
    queue_path.write_text(json.dumps(pending, indent=2))

    worker_script = ROOT / 'experiments' / 'pendulum-sweeps' / '_worker.py'
    assert worker_script.exists(), f"Worker script not found: {worker_script}"

    processes = []
    for i in range(min(NUM_WORKERS, len(pending))):
        device_arg = f"cuda:{i % max(1, GPU_COUNT)}" if GPU_COUNT > 0 else "cpu"
        cmd = [
            sys.executable, str(worker_script),
            '--queue-file', str(queue_path),
            '--output-dir', str(OUTPUT_DIR),
            '--worker-id', str(i),
            '--device', device_arg,
            '--total-tokens', str(TOTAL_TOKENS),
            '--batch-size', str(BATCH_SIZE),
            '--n-probe-traj', str(N_PROBE_TRAJECTORIES),
            '--cv-folds', str(CV_FOLDS),
            '--ridge-alphas', ','.join(str(a) for a in RIDGE_ALPHAS),
            '--test-frac', str(TEST_FRACTION),
            '--n-layers', str(N_LAYERS),
            '--n-heads', str(N_HEADS),
            '--log-every', str(LOG_EVERY),
        ]
        p = subprocess.Popen(cmd, cwd=str(ROOT))
        processes.append(p)
        print(f"Spawned worker {i} on {device_arg} (PID {p.pid})")

    print(f"\n{len(processes)} workers running. Waiting for completion...")
    for p in processes:
        p.wait()
    print("All workers finished.")


### Cell 6: Monitor Progress
Inspects completed jobs count in `pipeline_state.json`.

In [ ]:
if state_path.exists():
    state = json.loads(state_path.read_text())
    n_done = len(state.get('completed', []))
    print(f"Progress: {n_done}/{len(jobs)} jobs complete ({100*n_done/len(jobs):.1f}%)")
else:
    print("No state file found yet.")


### Cell 7: Aggregate Probe Results
Finds all per-job `d{W}_seed{S}_probes.csv` files and concatenates them into `probe_results_all.csv`.

In [ ]:
all_csvs = sorted(OUTPUT_DIR.rglob('*_probes.csv'))
if all_csvs:
    master_df = pd.concat([pd.read_csv(f) for f in all_csvs], ignore_index=True)
    master_csv = OUTPUT_DIR / 'probe_results_all.csv'
    master_df.to_csv(master_csv, index=False)
    print(f"Aggregated {len(master_df)} probe rows from {len(all_csvs)} files -> {master_csv.name}")
    display(master_df.head(20))
else:
    print("No probe CSVs found yet.")
